# Cannibalization: Naive vs. Fixed

## The Pair
- **Promoted Product (A):** `1127831` (Strawberries, National brand, 16 OZ)
- **Neighbor Product (B):** `860299` (Strawberries, National brand, 16 OZ)

**Why this pair?** Same sub-commodity, same brand tier. A plausible substitution story: if a customer wants National brand strawberries and A is on sale, they might buy A instead of B.

**Clean Promo Weeks:** Product A has 491 clean promo store-weeks across 24 distinct weeks and 106 stores. Plenty of data to evaluate.

## The Naive Result

A simple before/after comparison of A's promo weeks vs. all other weeks.

| Product | Avg Units During Promo | Avg Units Baseline | Naive % Change |
|---|---:|---:|---:|
| **A (1127831)** | 114.7 | 68.0 | **+68.6%** |
| **B (860299)** | 17.9 | 20.4 | **-12.6%** |

*Interpretation delayed. This is the raw, naive number. The goal is to break it.*

In [ ]:
# The Naive Comparison Query
con.sql("""
    WITH promo_weeks AS (
        SELECT DISTINCT week_no
        FROM product_store_week
        WHERE product_id = 1127831 AND is_price_promo AND NOT is_featured
    ),
    a_weekly AS (
        SELECT week_no, SUM(total_quantity) AS qty
        FROM product_store_week WHERE product_id = 1127831
        GROUP BY week_no
    ),
    b_weekly AS (
        SELECT week_no, SUM(total_quantity) AS qty
        FROM product_store_week WHERE product_id = 860299
        GROUP BY week_no
    )
    SELECT
        'A (1127831)' AS product,
        SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) AS total_units_during,
        SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) AS total_units_baseline,
        COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END) AS n_weeks_during,
        COUNT(CASE WHEN p.week_no IS NULL THEN 1 END) AS n_weeks_baseline,
        ROUND( (SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END), 0)) / 
               (SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NULL THEN 1 END), 0)) - 1, 4) AS pct_change
    FROM a_weekly a
    LEFT JOIN promo_weeks p ON a.week_no = p.week_no

    UNION ALL

    SELECT
        'B (860299)' AS product,
        SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) AS total_units_during,
        SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) AS total_units_baseline,
        COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END) AS n_weeks_during,
        COUNT(CASE WHEN p.week_no IS NULL THEN 1 END) AS n_weeks_baseline,
        ROUND( (SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END), 0)) / 
               (SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NULL THEN 1 END), 0)) - 1, 4) AS pct_change
    FROM b_weekly b
    LEFT JOIN promo_weeks p ON b.week_no = p.week_no
""").show()

## Why the Naive Result is Broken

Before attempting a fix, we ran diagnostics to identify the specific confounds.

1. **Seasonality:** Product A's promo weeks are highly clustered (weeks 2-13, 28-33, 54-70). Strawberries are highly seasonal. The naive baseline is comparing "strawberry season" to "winter."
2. **Baseline Contamination:** The "baseline" for B includes weeks where B itself was on promo. 99 out of 388 baseline rows (25.5%) were contaminated. This inflated B's baseline average.
3. **Store Composition:** 95 out of 106 stores overlap. Store mismatch is negligible.

**Conclusion:** The naive -12.6% drop for B is heavily biased by seasonality. The fix is a Difference-in-Differences (DiD) against an unaffected control product.

## The Fix: Difference-in-Differences (DiD) — and its failure

We introduced a control product (C: 1135476) that sold in 22 weeks.

### First DiD Attempt
| Metric | Avg During Promo | Avg Baseline | Change |
|---|---:|---:|---:|
| B (Neighbor) | 77.0 | 32.2 | +44.8 |
| C (Control) | 2.0 | 21.6 | -19.6 |

DiD result: **+64.4 units** (a halo effect, not cannibalization).

**The problem:** Product B was also on promotion in the same weeks as A. So B's +44.8 was its own promotion effect, not A's halo effect.

### Second Attempt: Clean DiD
We retried by restricting to store-weeks where B and C were at full price. The result: **NULL**.

**Why it failed:** During A's 24 promo weeks, control product C had **0 normal (non-promoted) store-weeks in 23 of those 24 weeks.** Product B showed the same pattern.

In [ ]:
# The Diagnostic that breaks the DiD
con.sql("""
    WITH a_promo_weeks AS (
        SELECT DISTINCT week_no FROM product_store_week
        WHERE product_id = 1127831 AND is_price_promo AND NOT is_featured
    )
    SELECT 
        apw.week_no,
        SUM(CASE WHEN psw.product_id = 860299 AND NOT psw.is_price_promo THEN 1 ELSE 0 END) AS b_normal_sw,
        SUM(CASE WHEN psw.product_id = 1135476 AND NOT psw.is_price_promo THEN 1 ELSE 0 END) AS c_normal_sw
    FROM a_promo_weeks apw
    LEFT JOIN product_store_week psw ON apw.week_no = psw.week_no AND psw.product_id IN (860299, 1135476)
    GROUP BY apw.week_no
    ORDER BY apw.week_no
""").show()

## Structural Finding: Category-Wide Promotions

The retailer runs category-wide promotions. When Strawberries go on sale, *all* Strawberry SKUs go on sale together. There are no "clean" weeks where A is promoted but B and C are at full price.

**Conclusion:** A binary promo DiD is mathematically valid but structurally invalid for this dataset. The treatment (price promo) is correlated across products at the category level. 

A future iteration would need to move from a binary "promo status" to continuous "discount depth" as the treatment variable, using a regression-based approach. This is a significant methodology change, not a simple fix.